Algorithmic cooling is an example of a quantum thermodynamics protocol where we effectively cool one qubt by shifting some of its entropy to other qubits. Below is the algorithm, that implements the unitary $U = \mathbb{1} - \left|100\right\rangle \left\langle100\right| - \left|011\right\rangle \left\langle011\right| + \left|100\right\rangle \left\langle011\right| + \left|011\right\rangle \left\langle100\right|$ on the iniitial state $\rho = \rho_b^{(1)} \otimes \rho_b^{(2)} \otimes\rho_b^{(3)}$, reducing the entropy of the first spin by swapping the state $\left|011\right\rangle$ and $\left|100\right\rangle$. 

In [15]:
import qiskit
import sympy as sp
print(qiskit.__version__)
print(sp.__version__)

2.5.2
1.14.0


In [2]:
eps = sp.symbols('epsilon', positive=True)
p0 = (1+eps)/2
p1 = (1-eps)/2

rho_b = sp.diag(p0,p1)
rho_b

Matrix([
[epsilon/2 + 1/2,               0],
[              0, 1/2 - epsilon/2]])

In [3]:
rho = sp.Matrix(sp.kronecker_product(sp.kronecker_product(rho_b,rho_b), rho_b))
rho.shape

(8, 8)

In [14]:
def idx(q1, q2, q3):
    return 4*q1 + 2*q2 + q3

def bits(i):
    return ((i>>2)&1, (i>>1)&1, i&1)

print(idx(1,0,0))
print(bits(4))

4
(1, 0, 0)


In [13]:
def cnot_matrix(control, target):
    M = sp.zeros(8,8)
    for i in range(8):
        b = list(bits(i))
        if b[control] == 1:
            b[target] ^= 1
        j = idx(*b)
        M[j,i] = 1
    return M

In [12]:
def toffoli_matrix(c1, c2, target):
    M = sp.zeros(8, 8)
    for i in range(8):
        b = list(bits(i))
        if b[c1] == 1 and b[c2] == 1:
            b[target] ^= 1
        j = idx(*b)
        M[j, i] = 1
    return M

In [7]:
U = cnot_matrix(0,1)
U = cnot_matrix(0,2) * U
U = toffoli_matrix(1,2,0) * U
U = cnot_matrix(0,1) * U
U = cnot_matrix(0,2) * U

In [8]:
rho_prime = sp.simplify(U * rho * U.T)

In [9]:
for i in range(8):
    b = bits(i)
    print(f"|{b[0]}{b[1]}{b[2]}>: {sp.factor(rho_prime[i,i])}")

|000>: (epsilon + 1)**3/8
|001>: -(epsilon - 1)*(epsilon + 1)**2/8
|010>: -(epsilon - 1)*(epsilon + 1)**2/8
|011>: -(epsilon - 1)*(epsilon + 1)**2/8
|100>: (epsilon - 1)**2*(epsilon + 1)/8
|101>: (epsilon - 1)**2*(epsilon + 1)/8
|110>: (epsilon - 1)**2*(epsilon + 1)/8
|111>: -(epsilon - 1)**3/8


In [11]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(3)
qc.cx(0,1)
qc.cx(0,2)
qc.ccx(1,2,0)
qc.cx(0,1)
qc.cx(0,2)

qc.draw(output='text')

┌───┐          
q_0: ──■────■──┤ X ├──■────■──
     ┌─┴─┐  │  └─┬─┘┌─┴─┐  │  
q_1: ┤ X ├──┼────■──┤ X ├──┼──
     └───┘┌─┴─┐  │  └───┘┌─┴─┐
q_2: ─────┤ X ├──■───────┤ X ├
          └───┘          └───┘